# Initial ML solution: efficient follow-up experiments

This notebook **reuses** the completed BM25 train/dev top-20 rankings and the BM25-evidence DistilBERT run. It does **not** rerun full-corpus dense retrieval or require dense train rankings. Run on a Colab GPU. Each costly cell has its own Drive output and can resume after interruption.

Experiments: (A) BM25 top-20 headroom and paired dense diagnostic; (B) same-model claim-only DistilBERT control; (C) gold-answer DistilBERT oracle; (D) zero-shot BGE cross-encoder reranking of BM25 top 20 on dev, then optionally train; (E) DistilBERT trained/evaluated on reranked top three; (F) validation-only Supported-threshold analysis and final comparison. The gold run is **not deployable**. The reranker cannot recover anything missing from BM25's top 20. Primary classification comparison remains macro-F1 and safety errors on the same official 500-claim dev split; retrieval remains URL-proxy hit@3 on the 386 judged claims. Reranker MRR, if reported, is computed over the saved top 20, not just the displayed top three.

Recommended order: setup, A, B, C, D-dev; inspect D-dev before deciding whether to run D-train and E. F is cheap. Keep run names/settings unchanged when resuming. Push this notebook and its two new scripts to GitHub before opening it in Colab.

In [ ]:
from pathlib import Path
import os, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
REPO_URL = 'https://github.com/muyishen2040/10718-ML-In-Practice.git'
REPO_DIR = Path('/content/10718-ML-In-Practice-followup')  # Leave the previous Colab checkout untouched.
if REPO_DIR.exists():
    dirty = subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip()
    if dirty:
        raise RuntimeError('Colab checkout has local edits; save them before pulling the new notebook/scripts.')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Repo revision:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
import json, gc, runpy, time
import torch

os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
HF_CACHE = Path('/content/huggingface_cache')
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['SENTENCE_TRANSFORMERS_HOME'] = str(HF_CACHE / 'sentence-transformers')
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, restart runtime, then rerun setup.')
print('GPU:', torch.cuda.get_device_name(0))
DATA_ROOT = Path('/content/drive/MyDrive/10718_averitec/v2_source_documents')
PROCESSED = DATA_ROOT / 'processed' / 'averitec'
BASE_RUN = DATA_ROOT / 'outputs' / 'initial_ml_solution_rerun_bs16'
FOLLOWUP = DATA_ROOT / 'outputs' / 'initial_ml_solution_followup_v1'
FOLLOWUP.mkdir(parents=True, exist_ok=True)
TRAIN_BM25 = DATA_ROOT / 'outputs/bm25/train_bm25_top20_v2/train_rankings.jsonl'
DEV_BM25 = DATA_ROOT / 'outputs/bm25/dev_bm25_top20_v2/dev_rankings.jsonl'
DEV_DENSE = BASE_RUN / 'dense/dev_bge_small_v15/dev_rankings.jsonl'
DEV_QRELS = PROCESSED / 'dev_qrels.jsonl'
BM25_TRANSFORMER = BASE_RUN / 'transformer/bm25_top3'
must_exist = [PROCESSED/'train.jsonl', PROCESSED/'dev.jsonl', TRAIN_BM25, DEV_BM25, DEV_QRELS, BM25_TRANSFORMER/'final_dev_predictions.jsonl', BM25_TRANSFORMER/'model_selection.json', BM25_TRANSFORMER/'transformer_run_config.json']
missing = [str(p) for p in must_exist if not p.is_file()]
if missing:
    raise FileNotFoundError('Missing previous experiment inputs:\n' + '\n'.join(missing))
for name in ('prepare_verifier_variant_rankings.py', 'run_cross_encoder_reranker.py', 'train_transformer_verifier.py'):
    if not (REPO_DIR / 'scripts' / name).is_file():
        raise FileNotFoundError(f'New script missing: {name}. Push local changes, then rerun setup/pull in Colab.')
print('Input checks passed. Follow-up outputs:', FOLLOWUP)


In [ ]:
def complete(paths):
    for path in map(Path, paths):
        if not path.is_file() or path.stat().st_size == 0:
            return False
        if path.suffix == '.json':
            try:
                json.loads(path.read_text(encoding='utf-8'))
            except (OSError, ValueError):
                return False
    return True

def run_script(label, script_name, args, outputs):
    if complete(outputs):
        print(label, ': complete; reusing saved artifacts.')
        return
    script = (REPO_DIR / 'scripts' / script_name).resolve()
    if not script.is_relative_to(REPO_DIR.resolve()) or not script.is_file():
        raise FileNotFoundError(script)
    old_argv = sys.argv[:]
    sys.argv = [str(script), *map(str, args)]
    print(f'\n=== {label} ===\nScript: {script}\nArguments: {args}', flush=True)
    start = time.monotonic()
    try:
        runpy.run_path(str(script), run_name='__main__')
    except KeyboardInterrupt:
        print('Interrupted; rerun this same cell to resume.', flush=True)
        raise
    finally:
        sys.argv = old_argv
        gc.collect()
        torch.cuda.empty_cache()
    if not complete(outputs):
        raise RuntimeError(f'{label} returned without complete artifacts: {outputs}')
    print(f'{label} finished in {(time.monotonic()-start)/60:.1f} minutes.')

# Shared paths/settings live here so any interrupted experiment cell can be
# resumed after rerunning only setup + this helper cell.
HEADROOM_PATH = FOLLOWUP / 'diagnostics/headroom.json'
VARIANTS = FOLLOWUP / 'variants'
CLAIM_ONLY = VARIANTS / 'claim_only'
GOLD = VARIANTS / 'gold'
TRANSFORMER_ROOT = FOLLOWUP / 'transformer'
COMMON_TRAIN_ARGS = ['--data-root', DATA_ROOT, '--output-dir', TRANSFORMER_ROOT, '--model', 'distilbert-base-uncased', '--max-length', '512', '--claim-token-budget', '96', '--evidence-token-budget', '110', '--validation-fraction', '0.10', '--learning-rates', '2e-5,5e-5', '--epoch-grid', '2,3', '--batch-size', '8', '--gradient-accumulation-steps', '2', '--checkpoint-steps', '100', '--seed', '718', '--resume']
baseline_model_config = json.loads((BM25_TRANSFORMER/'transformer_run_config.json').read_text())
PINNED_DISTILBERT_REV = baseline_model_config.get('resolved_model_revision') or baseline_model_config.get('requested_revision')
if PINNED_DISTILBERT_REV:
    COMMON_TRAIN_ARGS.extend(['--revision', PINNED_DISTILBERT_REV])
else:
    print('Warning: original DistilBERT revision was not recorded; new runs use the model ID only.')
def train_variant(name, train_rankings, dev_rankings):
    out = TRANSFORMER_ROOT / name
    args = [*COMMON_TRAIN_ARGS, '--run-name', name, '--train-rankings', train_rankings, '--eval-rankings', dev_rankings]
    run_script(f'DistilBERT / {name}', 'train_transformer_verifier.py', args, [out/'final_dev_metrics.json', out/'final_dev_predictions.jsonl', out/'transformer_manifest.json'])
    return out
RERANK_ROOT = FOLLOWUP / 'reranker'
RERANK_DEV = RERANK_ROOT / 'dev_bge_base_top20'
RERANK_TRAIN = RERANK_ROOT / 'train_bge_base_top20'
RERANK_SETTINGS = ['--output-dir', RERANK_ROOT, '--model', 'BAAI/bge-reranker-base', '--device', 'cuda', '--max-candidates', '20', '--max-seq-length', '384', '--batch-size', '8', '--checkpoint-every', '10', '--resume']


## A. Retrieval headroom (fast, no model)

Find the maximum URL-proxy hit@3 a reranker *could* attain from BM25 top 20. The optional dense comparison reports paired wins, not only mean differences. These qrels match source URLs, not necessarily decisive passage content.

In [ ]:
from src.utils.io import read_jsonl, write_json
from scripts.run_bm25 import load_qrels
qrels = {k: set(v) for k, v in load_qrels(DEV_QRELS).items() if v}
bm25 = {r['claim_id']: r for r in read_jsonl(DEV_BM25)}
judged = sorted(set(qrels) & set(bm25))
def ids(row, k):
    return {str(x['passage_id']) for x in row.get('retrieved', [])[:k]}
bm3 = {c for c in judged if ids(bm25[c], 3) & qrels[c]}
bm20 = {c for c in judged if ids(bm25[c], 20) & qrels[c]}
headroom = {'judged_claims': len(judged), 'bm25_hit_at_3': len(bm3)/len(judged), 'bm25_hit_at_20': len(bm20)/len(judged), 'additional_claims_recoverable_with_bm25_top20': len(bm20-bm3)}
if DEV_DENSE.is_file():
    dense = {r['claim_id']: r for r in read_jsonl(DEV_DENSE)}
    if set(dense) != set(bm25):
        raise ValueError('BM25 and dense claim sets differ')
    de3 = {c for c in judged if ids(dense[c], 3) & qrels[c]}
    union20 = {c for c in judged if (ids(bm25[c], 20) | ids(dense[c], 20)) & qrels[c]}
    headroom.update({'dense_hit_at_3': len(de3)/len(judged), 'bm25_only_top3_hits': len(bm3-de3), 'dense_only_top3_hits': len(de3-bm3), 'both_top3_hits': len(bm3&de3), 'union_top20_hit': len(union20)/len(judged)})
write_json(headroom, HEADROOM_PATH)
print(json.dumps(headroom, indent=2))
print('Reranker can reorder BM25 top 20 only; its hit@3 ceiling is bm25_hit_at_20.')


## B. Claim-only DistilBERT (controlled ablation)

Same pretrained model, train split, stratified 10% train-only validation, four trial settings, seed, and dev split as the completed BM25-evidence DistilBERT. Only evidence changes: every claim gets an empty evidence list. This is the key test of whether retrieved passages help this architecture. Resume by rerunning the interrupted cell.

In [ ]:
run_script('Prepare claim-only inputs', 'prepare_verifier_variant_rankings.py', ['--data-root', DATA_ROOT, '--output-dir', CLAIM_ONLY, '--mode', 'claim_only'], [CLAIM_ONLY/'train_rankings.jsonl', CLAIM_ONLY/'dev_rankings.jsonl', CLAIM_ONLY/'variant_manifest.json'])
CLAIM_ONLY_MODEL = train_variant('claim_only', CLAIM_ONLY/'train_rankings.jsonl', CLAIM_ONLY/'dev_rankings.jsonl')


## C. Gold-answer DistilBERT (oracle only)

Uses up to three annotated answer texts from each claim. These answers are not available to the deployed retriever. Compare with B to see whether a stronger evidence supply helps the same architecture, but never report this as end-to-end performance.

In [ ]:
run_script('Prepare gold-answer oracle inputs', 'prepare_verifier_variant_rankings.py', ['--data-root', DATA_ROOT, '--output-dir', GOLD, '--mode', 'gold'], [GOLD/'train_rankings.jsonl', GOLD/'dev_rankings.jsonl', GOLD/'variant_manifest.json'])
GOLD_MODEL = train_variant('gold_oracle', GOLD/'train_rankings.jsonl', GOLD/'dev_rankings.jsonl')


## D-dev. Zero-shot cross-encoder rerank of BM25 top 20

Fixed off-the-shelf `BAAI/bge-reranker-base`, one query-passage relevance logit, deterministic score descending then passage ID, at most 20 candidates per claim, user-facing k=3. This is the proposed efficient main retrieval method. The model never sees labels or qrels. Completed claims are appended to Drive every 10 claims; resume with the same run name and settings. Inspect hit@3 versus BM25 **before** deciding to run train reranking. See the [official BGE reranker model card](https://huggingface.co/BAAI/bge-reranker-base) for the query-passage scoring setup.

In [ ]:
run_script('BGE reranker / dev', 'run_cross_encoder_reranker.py', [*RERANK_SETTINGS, '--split', 'dev', '--run-name', 'dev_bge_base_top20', '--input-rankings', DEV_BM25, '--qrels', DEV_QRELS], [RERANK_DEV/'dev_rankings.jsonl', RERANK_DEV/'dev_metrics.json', RERANK_DEV/'dev_manifest.json'])
headroom = json.loads(HEADROOM_PATH.read_text()) if HEADROOM_PATH.is_file() else None
print('BM25 hit@3:', headroom['bm25_hit_at_3'] if headroom else 'Run A first')
print('Reranked hit@3:', json.loads((RERANK_DEV/'dev_metrics.json').read_text())['evidence_hit_at_3'])
print('BM25-top20 oracle ceiling:', headroom['bm25_hit_at_20'] if headroom else 'Run A first')


## D-audit. Inspect reranker wins/losses (fast)

Export up to ten cases where BM25 alone hits a proxy-relevant source and ten where reranking alone does. Read the passages manually before concluding the URL-proxy metric reflects decisive evidence. This is a post-evaluation diagnostic, not training data.

In [ ]:
from src.utils.io import read_jsonl, write_jsonl
from scripts.run_bm25 import load_qrels
qrels = {key: set(value) for key, value in load_qrels(DEV_QRELS).items() if value}
bm25 = {row['claim_id']: row for row in read_jsonl(DEV_BM25)}
reranked = {row['claim_id']: row for row in read_jsonl(RERANK_DEV/'dev_rankings.jsonl')}
audit = []
for claim_id in sorted(set(qrels) & set(bm25) & set(reranked)):
    b = bm25[claim_id]['retrieved'][:3]
    r = reranked[claim_id]['retrieved'][:3]
    b_hit = any(item['passage_id'] in qrels[claim_id] for item in b)
    r_hit = any(item['passage_id'] in qrels[claim_id] for item in r)
    if b_hit != r_hit:
        audit.append({'claim_id': claim_id, 'claim': bm25[claim_id]['claim'], 'proxy_winner': 'reranker' if r_hit else 'bm25', 'bm25_top3': b, 'reranker_top3': r})
selected_audit = [row for row in audit if row['proxy_winner']=='bm25'][:10] + [row for row in audit if row['proxy_winner']=='reranker'][:10]
audit_path = FOLLOWUP/'diagnostics/reranker_paired_audit.jsonl'
write_jsonl(selected_audit, audit_path)
print('BM25-only proxy hits:', sum(row['proxy_winner']=='bm25' for row in audit))
print('Reranker-only proxy hits:', sum(row['proxy_winner']=='reranker' for row in audit))
print('Manual-audit file:', audit_path)


## D-train and E. Main end-to-end method (optional)

Run these only if the D-dev retrieval result or manual evidence audit justifies it. Train reranking scores at most 20 saved BM25 passages per train claim (not millions of corpus chunks), then trains a *new* DistilBERT on those top-three passages. This keeps train and dev evidence construction matched. The reranker is zero-shot; only DistilBERT is supervised. Both cells resume.

In [ ]:
dev_reranker_config = json.loads((RERANK_DEV/'reranker_config.json').read_text())
RERANK_REV = dev_reranker_config.get('resolved_revision')
if not RERANK_REV:
    raise ValueError('Dev reranker revision was not recorded; cannot guarantee matched train/dev scoring.')
run_script('BGE reranker / train', 'run_cross_encoder_reranker.py', [*RERANK_SETTINGS, '--revision', RERANK_REV, '--split', 'train', '--run-name', 'train_bge_base_top20', '--input-rankings', TRAIN_BM25], [RERANK_TRAIN/'train_rankings.jsonl', RERANK_TRAIN/'train_metrics.json', RERANK_TRAIN/'train_manifest.json'])


In [ ]:
RERANK_MODEL = train_variant('bm25_reranked_top3', RERANK_TRAIN/'train_rankings.jsonl', RERANK_DEV/'dev_rankings.jsonl')


## F. Validation-only Supported threshold (fast, exploratory)

Using the *selected BM25 DistilBERT trial's train-only validation predictions*, test fixed Supported-probability thresholds. If argmax says Supported but its Supported probability is below the threshold, switch to the highest-probability non-Supported label. Pick by validation macro-F1, breaking ties by fewer false reassurances and then fewer false alarms. Apply **only the selected threshold** to official dev. This reuses the model-selection holdout, so regard it as an exploratory operating-point analysis, not independently validated calibration or a deployment policy. False reassurance/alarm rates in the standard metrics use all 500 claims as denominator; conditional rates are reported separately.

In [ ]:
from src.evaluation.classification import evaluate_predictions
from src.data.averitec import CANONICAL_LABELS
from src.utils.io import write_jsonl
selection = json.loads((BM25_TRANSFORMER/'model_selection.json').read_text())
val_path = Path(selection['selected_trial']['predictions_path'])
if not val_path.is_file():
    raise FileNotFoundError(f'Validation predictions unavailable: {val_path}')
val_rows = read_jsonl(val_path)
dev_rows = read_jsonl(BM25_TRANSFORMER/'final_dev_predictions.jsonl')
def threshold_predictions(rows, threshold):
    output = []
    for row in rows:
        copy = dict(row)
        probs = row['probabilities']
        if copy['predicted_label'] == 'Supported' and probs['Supported'] < threshold:
            copy['predicted_label'] = max((x for x in CANONICAL_LABELS if x != 'Supported'), key=lambda x: probs[x])
        copy['original_probabilities'] = copy.pop('probabilities')  # Do not report argmax ECE for thresholded decisions.
        output.append(copy)
    return output
def conditional_errors(rows, metrics):
    n_supported = sum(row['true_label'] == 'Supported' for row in rows)
    n_other = len(rows)-n_supported
    errors = metrics['decision_error_rates']
    return {'false_reassurance_given_not_supported': errors['false_reassurance_count']/n_other, 'false_alarm_given_supported': errors['false_alarm_count']/n_supported}
grid = [0.0, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]
sweep = []
for threshold in grid:
    altered = threshold_predictions(val_rows, threshold)
    metrics = evaluate_predictions(altered)
    sweep.append({'threshold': threshold, 'macro_f1': metrics['macro_f1'], **metrics['decision_error_rates'], **conditional_errors(val_rows, metrics)})
selected = sorted(sweep, key=lambda x: (-x['macro_f1'], x['false_reassurance_count'], x['false_alarm_count'], x['threshold']))[0]
dev_adjusted = threshold_predictions(dev_rows, selected['threshold'])
dev_metrics = evaluate_predictions(dev_adjusted)
SAFETY_DIR = FOLLOWUP/'safety_threshold'
SAFETY_DIR.mkdir(parents=True, exist_ok=True)
write_jsonl(dev_adjusted, SAFETY_DIR/'selected_dev_predictions.jsonl')
write_json({'selected_on': 'internal train validation only', 'validation_predictions': str(val_path), 'validation_sweep': sweep, 'selected': selected, 'dev_metrics': dev_metrics, 'dev_conditional_errors': conditional_errors(dev_rows, dev_metrics), 'warning': 'Exploratory threshold on reused model-selection holdout; do not select threshold on dev.'}, SAFETY_DIR/'threshold_report.json')
print('Selected on train-only validation:', selected)
print('Official dev macro-F1:', dev_metrics['macro_f1'])
print('Official dev conditional safety errors:', conditional_errors(dev_rows, dev_metrics))


## G. Comparison for the assignment

Compare only end-to-end models in the primary table; show claim-only and gold-answer runs as diagnostics. All classifier scores are on the same official 500 dev claims, but gold answers are oracle evidence. The reranker scores are on the same 386 URL-proxy-judged dev claims as BM25. Report all four model-selection trials from each run's `model_selection.json`, not only the winner. Do not call a small dev gain statistically significant without a paired analysis.

In [ ]:
runs = [
    ('TF-IDF/logistic + BM25', DATA_ROOT/'outputs/tfidf_logreg/retrieved_train_to_dev_v2/retrieved_train_to_dev_metrics.json'),
    ('Qwen3-4B + BM25', DATA_ROOT/'outputs/local_llm/dev_qwen3_4b_bm25_top3_v2/classify_top3_metrics.json'),
    ('BM25 + DistilBERT', BM25_TRANSFORMER/'final_dev_metrics.json'),
    ('Claim-only DistilBERT', TRANSFORMER_ROOT/'claim_only/final_dev_metrics.json'),
    ('Gold-answer DistilBERT (oracle)', TRANSFORMER_ROOT/'gold_oracle/final_dev_metrics.json'),
    ('BM25 reranked + DistilBERT', TRANSFORMER_ROOT/'bm25_reranked_top3/final_dev_metrics.json'),
]
print(f'{"Run":37s} {"Macro-F1":>9s} {"Accuracy":>9s} {"False reass.":>13s} {"False alarm":>11s}')
for name, path in runs:
    if not path.is_file():
        print(f'{name:37s} not run')
        continue
    result = json.loads(path.read_text())
    safety = result['decision_error_rates']
    print(f'{name:37s} {result["macro_f1"]:9.4f} {result["accuracy"]:9.4f} {safety["false_reassurance_rate"]:13.4f} {safety["false_alarm_rate"]:11.4f}')
if (RERANK_DEV/'dev_metrics.json').is_file():
    print('Reranker URL-proxy hit@3:', json.loads((RERANK_DEV/'dev_metrics.json').read_text())['evidence_hit_at_3'])
headroom = json.loads(HEADROOM_PATH.read_text()) if HEADROOM_PATH.is_file() else None
print('BM25 URL-proxy hit@3:', headroom['bm25_hit_at_3'] if headroom else 'Run A first')
